## Causal Audit — DoWhy Refutation Tests

In [15]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from dowhy import CausalModel

df = pd.read_csv(
    "../Data/Processed/Hillstorm_prepocessed.csv"
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df["segment"].value_counts().to_frame("Customer Count"))
display(df.groupby("segment")["conversion"].agg(["count", "sum", "mean"]))

Dataset shape: (64000, 13)
Columns: ['recency', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend', 'Customer ID', 'history_floor']


,Customer Count
segment,
Womens E-Mail,21387
Mens E-Mail,21307
No E-Mail,21306


,count,sum,mean
segment,,,
Mens E-Mail,21307,267,0.012531
No E-Mail,21306,122,0.005726
Womens E-Mail,21387,189,0.008837


## Data validation

In [16]:
TREATMENT_COL = "segment"
OUTCOME_COL = "conversion"
CONTROL = "No E-Mail"
TREATMENT_GROUPS = ["Mens E-Mail", "Womens E-Mail"]

expected_groups = set(TREATMENT_GROUPS + [CONTROL])
actual_groups = set(df[TREATMENT_COL].dropna().astype(str).str.strip().unique())

print("Expected groups:", expected_groups)
print("Actual groups:", actual_groups)
print("Missing values in required columns:")
display(df[[TREATMENT_COL, OUTCOME_COL]].isna().sum().to_frame("Missing Values"))

assert expected_groups.issubset(actual_groups), "One or more expected campaign labels are missing. Check the source data labels."
assert df[OUTCOME_COL].dropna().isin([0, 1]).all(), "conversion must be coded as 0/1."
assert df[[TREATMENT_COL, OUTCOME_COL]].notna().all().all(), "Missing treatment/outcome values need investigation before the audit."

print("Validation passed.")
print("Exact duplicate rows (for investigation only):", df.duplicated().sum())

Expected groups: {'No E-Mail', 'Womens E-Mail', 'Mens E-Mail'}
Actual groups: {'No E-Mail', 'Womens E-Mail', 'Mens E-Mail'}
Missing values in required columns:


,Missing Values
segment,0
conversion,0


Validation passed.
Exact duplicate rows (for investigation only): 0


## Estimate the experimental effect

In [17]:
output_dir = Path("../Reports/causal_audit")
output_dir.mkdir(parents=True, exist_ok=True)

audit_summary = []
audit_details = {}

def audit_campaign(treatment_group, control_group=CONTROL):
    print("\n" + "=" * 65)
    print(f"{treatment_group} vs {control_group}")
    print("=" * 65)

    pair = df.loc[
        df[TREATMENT_COL].isin([treatment_group, control_group]),
        [TREATMENT_COL, OUTCOME_COL]
    ].copy()
    pair[TREATMENT_COL] = pair[TREATMENT_COL].astype(str).str.strip()
    pair = pair.dropna(subset=[TREATMENT_COL, OUTCOME_COL]).copy()
    pair["treatment"] = (pair[TREATMENT_COL] == treatment_group).astype(int)
    pair = pair[["treatment", OUTCOME_COL]].copy()

    treated = pair.loc[pair["treatment"] == 1, OUTCOME_COL]
    control = pair.loc[pair["treatment"] == 0, OUTCOME_COL]
    p_treated = treated.mean()
    p_control = control.mean()
    raw_ate = p_treated - p_control
    raw_se = np.sqrt(
        p_treated * (1 - p_treated) / len(treated)
        + p_control * (1 - p_control) / len(control)
    )

    print("Rows used:", len(pair))
    print("Treated customers:", len(treated), "| Conversion rate:", round(p_treated, 6))
    print("Control customers:", len(control), "| Conversion rate:", round(p_control, 6))
    print("Raw experimental ATE:", round(raw_ate, 6))
    print("Raw ATE (percentage points):", round(raw_ate * 100, 4))
    print("Approx. 95% CI (percentage points):",
          (round((raw_ate - 1.96 * raw_se) * 100, 4),
           round((raw_ate + 1.96 * raw_se) * 100, 4)))

    graph = """
    digraph {
        treatment -> conversion;
    }
    """

    model = CausalModel(
        data=pair,
        treatment="treatment",
        outcome=OUTCOME_COL,
        graph=graph
    )

    estimand = model.identify_effect()
    print("\nIdentified estimand:")
    print(estimand)

    estimate = model.estimate_effect(
        estimand,
        method_name="backdoor.linear_regression",
        target_units="ate"
    )
    dowhy_ate = float(estimate.value)
    print("DoWhy ATE:", round(dowhy_ate, 6))
    print("DoWhy ATE (percentage points):", round(dowhy_ate * 100, 4))
    print("Difference: DoWhy minus raw ATE:", round(dowhy_ate - raw_ate, 8))

    results = {}

    print("\n--- Random common cause ---")
    results["random_common_cause"] = model.refute_estimate(
        estimand, estimate,
        method_name="random_common_cause",
        num_simulations=100,
        random_seed=42
    )
    print(results["random_common_cause"])

    print("\n--- Placebo treatment ---")
    results["placebo_treatment"] = model.refute_estimate(
        estimand, estimate,
        method_name="placebo_treatment_refuter",
        placebo_type="permute",
        num_simulations=100,
        random_seed=42
    )
    print(results["placebo_treatment"])

    print("\n--- Data subset ---")
    results["data_subset"] = model.refute_estimate(
        estimand, estimate,
        method_name="data_subset_refuter",
        subset_fraction=0.90,
        num_simulations=100,
        random_seed=42
    )
    print(results["data_subset"])

    print("\n--- Unobserved confounder sensitivity ---")
    sensitivity_rows = []
    strengths = [0.01, 0.05, 0.10]
    for treatment_strength in strengths:
        for outcome_strength in strengths:
            sensitivity_result = model.refute_estimate(
                estimand, estimate,
                method_name="add_unobserved_common_cause",
                confounders_effect_on_treatment="binary_flip",
                confounders_effect_on_outcome="binary_flip",
                effect_strength_on_treatment=treatment_strength,
                effect_strength_on_outcome=outcome_strength,
                random_seed=42
            )
            sensitivity_rows.append({
                "treatment_strength_setting": treatment_strength,
                "outcome_strength_setting": outcome_strength,
                "result": str(sensitivity_result)
            })
            print(
                f"Treatment strength={treatment_strength}, "
                f"outcome strength={outcome_strength}\n{sensitivity_result}\n"
            )

    sensitivity_table = pd.DataFrame(sensitivity_rows)
    sensitivity_table.to_csv(
        output_dir / f"{treatment_group.lower().replace(' ', '_')}_sensitivity.csv",
        index=False
    )

    report_file = output_dir / f"{treatment_group.lower().replace(' ', '_')}_audit.txt"
    with report_file.open("w", encoding="utf-8") as f:
        f.write(f"Treatment: {treatment_group}\nControl: {control_group}\n")
        f.write(f"Rows used: {len(pair)}\n")
        f.write(f"Treated conversion rate: {p_treated:.8f}\n")
        f.write(f"Control conversion rate: {p_control:.8f}\n")
        f.write(f"Raw ATE: {raw_ate:.8f}\nDoWhy ATE: {dowhy_ate:.8f}\n")
        for name, result in results.items():
            f.write(f"\n--- {name} ---\n{result}\n")
        f.write("\n--- Unobserved confounder sensitivity ---\n")
        f.write(sensitivity_table.to_string(index=False))

    audit_summary.append({
        "treatment": treatment_group,
        "control": control_group,
        "rows": len(pair),
        "treated_conversion_rate": p_treated,
        "control_conversion_rate": p_control,
        "raw_ATE": raw_ate,
        "raw_ATE_pp": raw_ate * 100,
        "raw_CI_low_pp": (raw_ate - 1.96 * raw_se) * 100,
        "raw_CI_high_pp": (raw_ate + 1.96 * raw_se) * 100,
        "DoWhy_ATE": dowhy_ate,
        "DoWhy_ATE_pp": dowhy_ate * 100,
        "report_file": str(report_file)
    })
    audit_details[treatment_group] = {
        "model": model,
        "estimand": estimand,
        "estimate": estimate,
        "refutations": results,
        "sensitivity": sensitivity_table,
        "report_file": report_file
    }

    print("\nSaved report:", report_file)
    return audit_details[treatment_group]

## Run both campaign audits

In [18]:
import networkx as nx

if not hasattr(nx.algorithms, "d_separated"):
	from networkx.algorithms.d_separation import is_d_separator
	nx.algorithms.d_separated = is_d_separator

mens_audit = audit_campaign("Mens E-Mail")
womens_audit = audit_campaign("Womens E-Mail")

audit_summary = pd.DataFrame(audit_summary)
display(audit_summary.round(6))
audit_summary.to_csv(output_dir / "audit_summary.csv", index=False)
print("\nSummary saved to:", output_dir / "audit_summary.csv")


Mens E-Mail vs No E-Mail
Rows used: 42613
Treated customers: 21307 | Conversion rate: 0.012531
Control customers: 21306 | Conversion rate: 0.005726
Raw experimental ATE: 0.006805
Raw ATE (percentage points): 0.6805
Approx. 95% CI (percentage points): (np.float64(0.5), np.float64(0.861))

Identified estimand:
Estimand type: nonparametric-ate

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                     
────────────(E[conversion])
d[treatment]               
Estimand assumption 1, Unconfoundedness: If U→{treatment} and U→conversion then P(conversion|treatment,,U) = P(conversion|treatment,)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

linear_regression
{'control_value': 0, 'treatment_value': 1, 'test_significance': None, 'evaluate_effect_strength': False, 'confidence_intervals': False, 'target_units': 'ate', 'effect_modifiers': []}
DoWhy ATE: 0.006805
DoWhy ATE (percentag

,treatment,control,rows,treated_conversion_rate,control_conversion_rate,raw_ATE,raw_ATE_pp,raw_CI_low_pp,raw_CI_high_pp,DoWhy_ATE,DoWhy_ATE_pp,report_file
0,Mens E-Mail,No E-Mail,42613,0.012531,0.005726,0.006805,0.680501,0.500014,0.860987,0.006805,0.680501,..\Reports\causal_audit\mens_e-mail_audit.txt
1,Womens E-Mail,No E-Mail,42693,0.008837,0.005726,0.003111,0.311106,0.149865,0.472347,0.003111,0.311106,..\Reports\causal_audit\womens_e-mail_audit.txt



Summary saved to: ..\Reports\causal_audit\audit_summary.csv


## Refutation and sensitivity results in tables

In [ ]:
import re
from IPython.display import display

def extract_number(text, label):
    match = re.search(
        rf"{re.escape(label)}\s*:\s*([-+]?(?:\d*\.)?\d+(?:[eE][-+]?\d+)?)",
        str(text),
        flags=re.IGNORECASE
    )
    return float(match.group(1)) if match else np.nan

refutation_rows = []

for treatment_group, details in audit_details.items():
    original_effect = float(details["estimate"].value)

    for test_name, result in details["refutations"].items():
        result_text = str(result)
        new_effect = extract_number(result_text, "New effect")
        p_value = extract_number(result_text, "p value")

        refutation_rows.append({
            "treatment": treatment_group,
            "test": test_name,
            "original_effect": original_effect,
            "new_effect": new_effect,
            "original_effect_pp": original_effect * 100,
            "new_effect_pp": new_effect * 100,
            "change_pp": (new_effect - original_effect) * 100,
            "p_value": p_value
        })

refutation_summary = pd.DataFrame(refutation_rows)

print("Refutation test summary (effects are also shown in percentage points):")
display(refutation_summary.round(6))

refutation_summary.to_csv(output_dir / "refutation_summary.csv", index=False)
print("Saved:", output_dir / "refutation_summary.csv")


sensitivity_rows = []

for treatment_group, details in audit_details.items():
    original_effect = float(details["estimate"].value)

    for _, row in details["sensitivity"].iterrows():
        result_text = row["result"]
        new_effect = extract_number(result_text, "New effect")

        sensitivity_rows.append({
            "treatment": treatment_group,
            "treatment_strength_setting": row["treatment_strength_setting"],
            "outcome_strength_setting": row["outcome_strength_setting"],
            "original_effect": original_effect,
            "new_effect": new_effect,
            "original_effect_pp": original_effect * 100,
            "new_effect_pp": new_effect * 100,
            "change_pp": (new_effect - original_effect) * 100
        })

sensitivity_summary = pd.DataFrame(sensitivity_rows)

print("Unobserved-confounder sensitivity summary:")
display(sensitivity_summary.round(6))

sensitivity_summary.to_csv(output_dir / "sensitivity_summary.csv", index=False)
print("Saved:", output_dir / "sensitivity_summary.csv")


Refutation test summary (effects are also shown in percentage points):


,treatment,test,original_effect,new_effect,original_effect_pp,new_effect_pp,change_pp,p_value
0,Mens E-Mail,random_common_cause,0.006805,0.006806,0.680501,0.680576,0.000075,0.86
1,Mens E-Mail,placebo_treatment,0.006805,0.000082,0.680501,0.008218,-0.672283,0.98
2,Mens E-Mail,data_subset,0.006805,0.006826,0.680501,0.682603,0.002103,0.96
3,Womens E-Mail,random_common_cause,0.003111,0.003111,0.311106,0.311104,-0.000002,0.98
4,Womens E-Mail,placebo_treatment,0.003111,-0.000061,0.311106,-0.006137,-0.317243,0.98
5,Womens E-Mail,data_subset,0.003111,0.003073,0.311106,0.307309,-0.003797,1.00


Saved: ..\Reports\causal_audit\refutation_summary.csv
Unobserved-confounder sensitivity summary:


,treatment,treatment_strength_setting,outcome_strength_setting,original_effect,new_effect,original_effect_pp,new_effect_pp,change_pp
0,Mens E-Mail,0.01,0.01,0.006805,0.007392,0.680501,0.739155,0.058655
1,Mens E-Mail,0.01,0.05,0.006805,0.006450,0.680501,0.644958,-0.035543
2,Mens E-Mail,0.01,0.10,0.006805,0.001458,0.680501,0.145813,-0.534688
3,Mens E-Mail,0.05,0.01,0.006805,0.006997,0.680501,0.699651,0.019150
4,Mens E-Mail,0.05,0.05,0.006805,0.007797,0.680501,0.779710,0.099210
5,Mens E-Mail,0.05,0.10,0.006805,0.002754,0.680501,0.275401,-0.405099
6,Mens E-Mail,0.10,0.01,0.006805,0.006178,0.680501,0.617764,-0.062737
7,Mens E-Mail,0.10,0.05,0.006805,0.006640,0.680501,0.663967,-0.016533
8,Mens E-Mail,0.10,0.10,0.006805,0.011026,0.680501,1.102570,0.422069
9,Womens E-Mail,0.01,0.01,0.003111,0.004566,0.311106,0.456595,0.145489


Saved: ..\Reports\causal_audit\sensitivity_summary.csv


## Conclusion

The causal audit was conducted to estimate the effect of email campaigns on customer conversion and assess the robustness of the estimated effects using DoWhy refutation and sensitivity tests.

The **Men's Email campaign** showed an estimated Average Treatment Effect (ATE) of **0.006805**, equivalent to an increase of approximately **0.68 percentage points** in conversion compared with the No Email group. The **Women's Email campaign** showed an estimated ATE of **0.003111**, equivalent to an increase of approximately **0.31 percentage points**.

The random common cause and data subset refutation tests produced estimates close to the original effects for both campaigns, indicating that the estimates were relatively stable under these tests. The placebo treatment tests produced effects close to zero, as expected when the original treatment effect is not present in the placebo treatment.

However, the unobserved-confounder sensitivity analysis showed that the estimated effects can change under some hypothetical confounder settings. Therefore, the refutation tests do not prove that hidden confounding is absent.

**Overall,** both campaigns showed positive estimated effects on conversion in this experiment, with the Men's Email campaign showing the larger estimated effect. The results support the robustness of the estimates under the specific tests performed, while the sensitivity analysis highlights the importance of considering potential unobserved confounding. These findings should be interpreted in the context of the experiment and the assumptions of the causal analysis.